In [ ]:
# In[0]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.2 MB/s eta 0:00:00


In [ ]:
# In[1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score, precision_recall_curve, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import gensim
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# In[2]:
# Cargar los archivos

train_df = pd.read_csv("train_final.csv")
test_df = pd.read_csv("test_final.csv")

X_train_raw = train_df["Headline"].astype(str).tolist()
y_train = train_df["Category"].astype(int).values

X_test_raw = test_df["Headline"].astype(str).tolist()
y_test = test_df["Category"].astype(int).values

# Tokenizador simple
def tokenizer(text):
    return text.lower().split()


In [ ]:
# In[3]:
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

counter = Counter()
for tokens in yield_tokens(X_train_raw):
    counter.update(tokens)

vocab = {"<unk>":0, "<pad>":1}
for idx, word in enumerate(counter.keys(), start=2):
    vocab[word] = idx

pad_idx = vocab["<pad>"]


In [ ]:
# In[4]:
def encode_text(text):
    return [vocab.get(token, vocab["<unk>"]) for token in tokenizer(text)]

X_train_enc = [encode_text(t) for t in X_train_raw]
X_test_enc = [encode_text(t) for t in X_test_raw]

X_train_tensor = pad_sequence([torch.tensor(seq) for seq in X_train_enc], batch_first=True, padding_value=pad_idx)
X_test_tensor = pad_sequence([torch.tensor(seq) for seq in X_test_enc], batch_first=True, padding_value=pad_idx)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


In [ ]:
# In[5]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
# In[6]:
# Descargar embeddings fastText en español directamente desde la fuente oficial
!wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz
!gunzip -f cc.es.300.vec.gz

print("Cargando embeddings fastText en español...")
fasttext_model = gensim.models.KeyedVectors.load_word2vec_format("cc.es.300.vec")

embedding_dim = fasttext_model.vector_size
embedding_matrix = torch.zeros((len(vocab), embedding_dim))
for word, idx in vocab.items():
    if word in fasttext_model:
        embedding_matrix[idx] = torch.tensor(fasttext_model[word])
    else:
        embedding_matrix[idx] = torch.randn(embedding_dim) * 0.01


Cargando embeddings fastText en español...


In [ ]:
import torch
import pickle

# 1. Guardar vocabulario (diccionario de tokens → índices)
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# 2. Guardar tensores de entrenamiento y test
torch.save(X_train_tensor, "X_train.pt")
torch.save(y_train_tensor, "y_train.pt")
torch.save(X_test_tensor, "X_test.pt")
torch.save(y_test_tensor, "y_test.pt")

# 3. Guardar embedding_matrix ya construida para tu vocabulario
torch.save(embedding_matrix, "embedding_matrix.pt")

print("Guardado completo: vocab, tensores train/test y embedding_matrix.")



Guardado completo: vocab, tensores train/test y embedding_matrix.
